# FLUX Scaling Animation

Generate animation frame sequences by progressively scaling an embedding.

1. Load T5 and CLIP embeddings
2. Pick which embedding to scale (T5, CLIP, or Both)
3. Set a scale factor and number of increments
4. Generate one image per increment, saving frames to `data/sequence/scale_<factor>_<timestamp>/`

In [ ]:
import torch
import json
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import os
from pathlib import Path
from datetime import datetime

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load models path from config
current_dir = Path.cwd()
models_path_file = current_dir.parent / "misc/paths/models.txt"
with open(models_path_file, 'r') as f:
    models_path = f.read().strip()
MODELS_DIR = current_dir.parent / models_path

T5_MODEL_PATH = os.path.join(MODELS_DIR, "t5-v1_1-xxl")
CLIP_MODEL_PATH = os.path.join(MODELS_DIR, "clip")
FLUX_MODEL_PATH = os.path.join(MODELS_DIR, "FLUX.1-schnell")

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Models directory: {os.path.abspath(MODELS_DIR)}")
print(f"T5 path: {os.path.abspath(T5_MODEL_PATH)}")
print(f"FLUX path: {os.path.abspath(FLUX_MODEL_PATH)}")

In [ ]:
# Load Hugging Face token from file
token_file = current_dir.parent / "misc/credentials/hf.txt"
print(f"Looking for HF token at: {token_file}")

if token_file.exists():
    with open(token_file, 'r') as f:
        hf_token = f.read().strip()
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token)

# Load FLUX
from diffusers import FluxPipeline

try:
    if not os.path.exists(FLUX_MODEL_PATH):
        flux_pipe = FluxPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-schnell",
            torch_dtype=torch.bfloat16
        )
        flux_pipe.save_pretrained(FLUX_MODEL_PATH)
    else:
        flux_pipe = FluxPipeline.from_pretrained(
            FLUX_MODEL_PATH,
            torch_dtype=torch.bfloat16,
            local_files_only=True
        )
    flux_pipe = flux_pipe.to(device)
except Exception as e:
    print(f"Error loading FLUX: {e}")

In [ ]:
# ==================== EMBEDDING LOADING ====================

T5_EMBEDDINGS_DIR = current_dir.parent / "data/embeddings/T5/"
CLIP_EMBEDDINGS_DIR = current_dir.parent / "data/embeddings/CLIP/"

# Global variables for loaded embeddings
loaded_t5_embedding = None
loaded_clip_embedding = None
loaded_t5_prompt = None
loaded_clip_prompt = None

# ==================== T5 EMBEDDING WIDGETS ====================

t5_file_dropdown = widgets.Dropdown(
    options=[],
    description='T5 Embedding:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

load_t5_button = widgets.Button(
    description='Load T5',
    button_style='success'
)

t5_load_output = widgets.Output()

if T5_EMBEDDINGS_DIR.exists():
    t5_files = sorted([str(f.relative_to(T5_EMBEDDINGS_DIR)) for f in T5_EMBEDDINGS_DIR.glob('**/*.json')])
    t5_file_dropdown.options = t5_files

def load_t5_embedding_file(b):
    global loaded_t5_embedding, loaded_t5_prompt
    with t5_load_output:
        t5_load_output.clear_output()
        filename = t5_file_dropdown.value
        if not filename:
            print("No file selected!")
            return
        filepath = T5_EMBEDDINGS_DIR / filename
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
            loaded_t5_embedding = np.array(data['embedding'])
            loaded_t5_prompt = data.get('prompt', 'Unknown')
            print(f"Loaded T5 embedding: {filename}")
            print(f"  Prompt: '{loaded_t5_prompt}'")
            print(f"  Shape: {loaded_t5_embedding.shape}")
        except Exception as e:
            print(f"Error loading T5 embedding: {e}")

load_t5_button.on_click(load_t5_embedding_file)

# ==================== CLIP EMBEDDING WIDGETS ====================

clip_file_dropdown = widgets.Dropdown(
    options=[],
    description='CLIP Embedding:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

load_clip_button = widgets.Button(
    description='Load CLIP',
    button_style='success'
)

clip_load_output = widgets.Output()

if CLIP_EMBEDDINGS_DIR.exists():
    clip_files = sorted([str(f.relative_to(CLIP_EMBEDDINGS_DIR)) for f in CLIP_EMBEDDINGS_DIR.glob('**/*.json')])
    clip_file_dropdown.options = clip_files

def load_clip_embedding_file(b):
    global loaded_clip_embedding, loaded_clip_prompt
    with clip_load_output:
        clip_load_output.clear_output()
        filename = clip_file_dropdown.value
        if not filename:
            print("No file selected!")
            return
        filepath = CLIP_EMBEDDINGS_DIR / filename
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
            loaded_clip_embedding = np.array(data['embedding'])
            loaded_clip_prompt = data.get('prompt', 'Unknown')
            print(f"Loaded CLIP embedding: {filename}")
            print(f"  Prompt: '{loaded_clip_prompt}'")
            print(f"  Shape: {loaded_clip_embedding.shape}")
        except Exception as e:
            print(f"Error loading CLIP embedding: {e}")

load_clip_button.on_click(load_clip_embedding_file)

# ==================== DISPLAY ====================

display(widgets.VBox([
    widgets.HTML("<h3>1. T5 Embedding (Required)</h3>"),
    t5_file_dropdown,
    load_t5_button,
    t5_load_output,
    widgets.HTML("<br><h3>2. CLIP Embedding (Optional - auto-generated if not loaded)</h3>"),
    clip_file_dropdown,
    load_clip_button,
    clip_load_output,
]))

In [ ]:
# ==================== ANIMATION CONTROLS & GENERATION ====================

SEQUENCE_DIR = current_dir.parent / "data/sequence"
os.makedirs(SEQUENCE_DIR, exist_ok=True)

# Which embedding to scale
scale_target = widgets.RadioButtons(
    options=['T5', 'CLIP', 'Both'],
    value='T5',
    description='Scale:',
    style={'description_width': 'initial'}
)

# Scale factor per step
scale_factor_input = widgets.FloatText(
    value=1.01,
    description='Scale factor per step:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Number of frames
num_frames_input = widgets.IntText(
    value=100,
    description='Number of frames:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Generation settings
seed_input = widgets.IntText(
    value=42,
    description='Seed:',
    style={'description_width': 'initial'}
)

steps_input = widgets.IntSlider(
    value=4,
    min=1,
    max=50,
    step=1,
    description='Inference Steps:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

width_input = widgets.IntText(
    value=512,
    description='Width:',
    style={'description_width': 'initial'}
)

height_input = widgets.IntText(
    value=512,
    description='Height:',
    style={'description_width': 'initial'}
)

generate_button = widgets.Button(
    description='Generate Animation Frames',
    button_style='primary',
    layout=widgets.Layout(width='300px', height='50px')
)

generation_output = widgets.Output()


def generate_animation_frames(b):
    with generation_output:
        generation_output.clear_output()

        if 'flux_pipe' not in globals():
            print("FLUX not loaded!")
            return

        if loaded_t5_embedding is None:
            print("No T5 embedding loaded! Load a T5 embedding first.")
            return

        target = scale_target.value
        factor = scale_factor_input.value
        num_frames = num_frames_input.value

        if num_frames < 1:
            print("Number of frames must be at least 1.")
            return

        # Create output directory
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = SEQUENCE_DIR / f"scale_{factor}_{timestamp}"
        os.makedirs(output_dir, exist_ok=True)

        print(f"Generating {num_frames} frames")
        print(f"  Scale target: {target}")
        print(f"  Scale factor per step: {factor}")
        print(f"  Output dir: {output_dir}")
        print(f"  Seed: {seed_input.value}")
        print(f"  Steps: {steps_input.value}")
        print(f"  Dimensions: {width_input.value}x{height_input.value}")
        print()

        # Prepare base CLIP pooled embeddings (computed once)
        if loaded_clip_embedding is not None:
            base_clip_pooled = torch.from_numpy(
                loaded_clip_embedding[-1:].astype(np.float32)
            ).to(device=device, dtype=torch.bfloat16)
        else:
            # Auto-generate from T5 prompt
            if loaded_t5_prompt:
                print("No CLIP embedding loaded, generating from T5 prompt...")
                with torch.no_grad():
                    _, base_clip_pooled, _ = flux_pipe.encode_prompt(
                        prompt=loaded_t5_prompt,
                        prompt_2=None,
                        device=device,
                        num_images_per_prompt=1,
                        max_sequence_length=512,
                    )
            else:
                print("No CLIP embedding and no prompt available!")
                return

        # Prepare base T5 tensor (computed once)
        base_t5_tensor = torch.from_numpy(
            loaded_t5_embedding.astype(np.float32)
        ).to(device=device, dtype=torch.bfloat16).unsqueeze(0)

        # Generate frames
        for i in range(num_frames):
            cumulative_scale = factor ** i

            # Apply scaling to the chosen embedding(s)
            if target == 'T5' or target == 'Both':
                t5_tensor = base_t5_tensor * cumulative_scale
            else:
                t5_tensor = base_t5_tensor

            if target == 'CLIP' or target == 'Both':
                pooled_embeds = base_clip_pooled * cumulative_scale
            else:
                pooled_embeds = base_clip_pooled

            try:
                image = flux_pipe(
                    prompt_embeds=t5_tensor,
                    pooled_prompt_embeds=pooled_embeds,
                    num_inference_steps=steps_input.value,
                    guidance_scale=0.0,
                    height=height_input.value,
                    width=width_input.value,
                    generator=torch.manual_seed(seed_input.value)
                ).images[0]

                frame_path = output_dir / f"frame_{i:04d}.png"
                image.save(frame_path)
                print(f"  Frame {i:04d}/{num_frames-1} saved (scale: {cumulative_scale:.6f})")

            except Exception as e:
                print(f"  Error on frame {i}: {e}")
                import traceback
                traceback.print_exc()
                break

        print(f"\nDone! {num_frames} frames saved to {output_dir}")


generate_button.on_click(generate_animation_frames)

# ==================== DISPLAY ====================

display(widgets.VBox([
    widgets.HTML("<h2>Scaling Animation</h2>"),

    widgets.HTML("<h3>Animation Settings</h3>"),
    scale_target,
    scale_factor_input,
    num_frames_input,

    widgets.HTML("<br><h3>Generation Settings</h3>"),
    seed_input,
    steps_input,
    widgets.HTML("<b>Image Dimensions:</b>"),
    widgets.HBox([width_input, height_input]),
    widgets.HTML("<br>"),
    generate_button,

    widgets.HTML("<br><h3>Output</h3>"),
    generation_output
]))

---
<sub>Latent Vandalism Workshop &bull; Laura Wagner, 2026 &bull; [laurajul.github.io](https://laurajul.github.io/)</sub>